# Download VisText Data

In [15]:
# https://drive.google.com/file/d/16SV-ahp5UCCuXgnZRNfHSvMJJeYsTxUO/view?usp=sharing
# https://drive.google.com/file/d/1h9csLbRIb_zfuGy6ezCgBqkBdlAjnS_U/view?usp=sharing

!gdown --id "1h9csLbRIb_zfuGy6ezCgBqkBdlAjnS_U"
!gdown --id "16SV-ahp5UCCuXgnZRNfHSvMJJeYsTxUO"

!unzip /kaggle/working/ground-truth.zip

!unzip /kaggle/working/sub-image.zip

/Users/ali/miniconda3/lib/python3.13/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1h9csLbRIb_zfuGy6ezCgBqkBdlAjnS_U
To: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/sub-image.zip
100%|██████████████████████████████████████| 4.92M/4.92M [00:01<00:00, 2.51MB/s]
/Users/ali/miniconda3/lib/python3.13/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=16SV-ahp5UCCuXgnZRNfHSvMJJeYsTxUO
To: /Users/ali/Desktop/RA/Wageningen/Github/Uploading/V12/Soil-Health/VisText/Evaluation/نتیجه گرفتن/Inferance/ground-truth.zip
100%|█████████████████████████

# ZeroShot prompt

In [29]:
import base64
import json
import os
import time
from pathlib import Path
from typing import Optional, Dict, List
import requests
from PIL import Image

# =========================
# 1) CONFIGURATION (AVALAI)
# =========================
AVALAI_API_KEY = os.getenv("AVALAI_API_KEY")
if not AVALAI_API_KEY:
    AVALAI_API_KEY = input("Enter AVALAI_API_KEY (or set env var AVALAI_API_KEY): ").strip()
if not AVALAI_API_KEY:
    raise RuntimeError("Missing API key. Set env var AVALAI_API_KEY or provide it when prompted.")

MODEL = "gemini-2.5-flash"                 # ← مدلی که می‌خوای استفاده کنی
BASE_URL = "https://api.avalai.ir/v1"          # ← بیس URL ای AVALAI

# ---- THREE PATHS ----
IMAGES_ROOT = "./sub_image_1000"        # 1) folder with images (relative to notebook)
LABELS_ROOT = "./labels_1000"     # 2) folder with JSON labels (72.json etc.)
OUTPUT_DIR  = "./vistext_Zeroshot_outputs"  # 3) output TTL folder (relative)

# ---- RATE LIMITING ----
REQUEST_DELAY = 2.0  # seconds between successful API calls (to respect rate limits)

# Optional flags
SKIP_EXISTING = True             # skip if TTL already exists
USE_VIS_TEXT_CONTEXT = False     # if True, send caption/scenegraph/datatable from JSON

WRITE_MANIFEST = True
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

VALID_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# =========================
# 2) PROMPT
# =========================
PROMPT_TEMPLATE = r"""
System
You are a chart-to-RDF extractor. Given ONE chart image, output ONLY valid RDF/Turtle (TTL) describing the chart structure and the datapoints that are visible or can be reasonably read from the chart axes.

The dataset contains many chart types (bar/stacked_bar/line/area/scatter/pie/histogram/boxplot/heatmap/table/other). Your primary objective is HIGH RECALL without hallucinating: extract as many correct categories/series and datapoints as possible. If a value is not explicitly labeled but can be estimated from the axis ticks/gridlines, you MUST still output it and mark it as estimated.

========================
1) Output format (STRICT)
========================
- Output Turtle only. No prose, no explanations, no code fences, no comments.
- Use ONLY the prefixes below. Do NOT introduce new prefixes.
- Ensure syntactically valid Turtle.
- Escape quotes inside literals.
- Collapse extra whitespace in extracted text.

=================
2) Prefixes (ONLY)
=================
@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

================================
3) Deterministic identifiers (ONLY)
================================
Use the numeric image id {IMG_ID} inside IRIs exactly like this:
- Chart:      ex:chart-{IMG_ID}
- X axis:     ex:x-{IMG_ID}
- Y axis:     ex:y-{IMG_ID}
- Series:     ex:series-{IMG_ID}-{N}          (N = 1,2,3,...)
- Bars:       ex:mark-{IMG_ID}-{N}-{M}        (series N, bar index M)
- Points:     ex:point-{IMG_ID}-{N}-{M}       (series N, point index M)
- Slices:     ex:slice-{IMG_ID}-{N}-{M}       (series N, slice index M)
- Cells:      ex:cell-{IMG_ID}-{M}            (heatmap cell index M)
- Bins:       ex:bin-{IMG_ID}-{M}             (histogram bin index M)
- Stats:      ex:stats-{IMG_ID}-{N}-{M}       (boxplot stats objects)

Do not invent other naming patterns.

=================================
4) REQUIRED base graph (ALWAYS)
=================================
You MUST output this skeleton, filling literals when readable:

ex:chart-{IMG_ID} a chart:Chart ;
    chart:chartType "bar"|"stacked_bar"|"line"|"area"|"scatter"|"pie"|"histogram"|"boxplot"|"heatmap"|"table"|"other" ;
    chart:xAxis ex:x-{IMG_ID} ;
    chart:yAxis ex:y-{IMG_ID} .

ex:x-{IMG_ID} a chart:Axis ;
    chart:label "" ;
    chart:scaleMin "" ;
    chart:scaleMax "" .

ex:y-{IMG_ID} a chart:Axis ;
    chart:label "" ;
    chart:scaleMin "" ;
    chart:scaleMax "" .

# Always create at least one series:
ex:series-{IMG_ID}-1 a chart:Series .

========================================
5) Extract chart-level fields (if visible)
========================================
- If a title is visible at the top, add:
  ex:chart-{IMG_ID} chart:title "..." .
- If a subtitle/source is visible, you MAY add:
  ex:chart-{IMG_ID} chart:subtitle "..." .
  ex:chart-{IMG_ID} chart:source "..." .

Axis labels:
- If x-axis label text is readable, set:
  ex:x-{IMG_ID} chart:label "..." .
- If y-axis label text is readable, set:
  ex:y-{IMG_ID} chart:label "..." .

Axis min/max:
- If smallest and largest tick labels on an axis are readable, set scaleMin/scaleMax.
- Use plain strings (no commas). If unsure, leave "".

=========================
6) Series / legend handling
=========================
- If there is a legend or multiple series labels (colors/line styles):
  - Create one series per legend entry: ex:series-{IMG_ID}-1, -2, ...
  - If legend text is readable, add on each series:
    chart:seriesName "..." .
- If no legend is visible, keep only series-1.

==============================
7) Datapoints: general policies
==============================
A) High-recall rule (IMPORTANT)
- If the chart clearly contains datapoints, you MUST output datapoints.
- Do NOT output an empty datapoint set just because values are not printed.
- If you can estimate values from axis ticks/gridlines with reasonable confidence, output them and mark them as estimated.

B) No hallucination rule
- Do not invent categories, legend labels, or datapoints that are not supported by visible marks.
- If you truly cannot read a category label, you MAY use a short placeholder like "UNREADABLE_CATEGORY_{M}" ONLY if the bar/mark is clearly present, but prefer readable labels.

C) Estimation metadata (use when needed)
When a numeric value is estimated (not explicitly printed), add:
  chart:valueEstimated "true"^^xsd:boolean .
And optionally:
  chart:confidence "0.3"^^xsd:decimal  (0.0 to 1.0)

When the numeric value is explicitly printed (e.g., data label), set:
  chart:valueEstimated "false"^^xsd:boolean .

==========================
8) Datapoints by chart type
==========================

8.1) BAR / STACKED_BAR
If chartType is "bar" or "stacked_bar":
- For each visible bar, create:

ex:mark-{IMG_ID}-{N}-{M} a chart:Bar ;
    chart:series   ex:series-{IMG_ID}-{N} ;
    chart:category "..." ;
    chart:value    "..."^^xsd:decimal ;
    chart:valueEstimated "true"^^xsd:boolean| "false"^^xsd:boolean .

Rules:
- chart:category: use the visible category label.
- chart:value:
  - If value is printed, copy it.
  - Else estimate from y-axis ticks/gridlines (prefer 1–2 decimal places).
- If stacked bars: still output one Bar per segment IF segment boundaries are visible and values can be read/estimated. Otherwise output one Bar per category for the total, and set:
  ex:chart-{IMG_ID} chart:stacked "true"^^xsd:boolean .

8.2) LINE / AREA
If chartType is "line" or "area":
- Extract representative points aligned with labeled x-axis ticks (start/end + major ticks + peaks/troughs).
- Aim for 6–12 points per series if possible.
- Each point:

ex:point-{IMG_ID}-{N}-{M} a chart:Point ;
    chart:series ex:series-{IMG_ID}-{N} ;
    chart:xVal   "..." ;
    chart:yVal   "..."^^xsd:decimal ;
    chart:valueEstimated "true"^^xsd:boolean| "false"^^xsd:boolean .

xVal typing:
- If x looks like a year (e.g., 2006): use plain string "2006" (do NOT use extra datatypes/prefixes).
- If x looks like month-year: use plain string like "2018-01".
- Else use the visible category text.

yVal:
- Use printed values if present; else estimate from y-axis ticks.

8.3) SCATTER
If chartType is "scatter":
- Extract as many points as possible (at least 10 if the plot is dense, otherwise all visible).
- Each point uses chart:xVal (string) and chart:yVal (decimal) like line charts.
- Estimate values from axes if needed and mark valueEstimated accordingly.

8.4) PIE / DONUT
If chartType is "pie":
- For each visible slice with readable label and/or percentage:
ex:slice-{IMG_ID}-{N}-{M} a chart:Slice ;
    chart:series ex:series-{IMG_ID}-{N} ;
    chart:category "..." ;
    chart:value "..."^^xsd:decimal ;
    chart:valueUnit "percent" ;
    chart:valueEstimated "true"^^xsd:boolean| "false"^^xsd:boolean .

Rules:
- If percent is printed, use it and valueEstimated=false.
- Else estimate percent from relative slice size ONLY if it is visually clear; otherwise omit that slice.

8.5) HISTOGRAM
If chartType is "histogram":
- For each visible bin:
ex:bin-{IMG_ID}-{M} a chart:Bin ;
    chart:category "BIN_{M}" ;
    chart:xRange "..." ;
    chart:value "..."^^xsd:decimal ;
    chart:valueEstimated "true"^^xsd:boolean| "false"^^xsd:boolean .

xRange example: "10–20" or "0–5".
If xRange text is not readable, leave it as "" but still include the bin if its bar is visible.

8.6) BOXPLOT
If chartType is "boxplot":
- For each group, extract five-number summary if readable/estimable from axis:
ex:stats-{IMG_ID}-{N}-{M} a chart:BoxStats ;
    chart:series ex:series-{IMG_ID}-{N} ;
    chart:category "..." ;
    chart:min "..."^^xsd:decimal ;
    chart:q1 "..."^^xsd:decimal ;
    chart:median "..."^^xsd:decimal ;
    chart:q3 "..."^^xsd:decimal ;
    chart:max "..."^^xsd:decimal ;
    chart:valueEstimated "true"^^xsd:boolean| "false"^^xsd:boolean .

8.7) HEATMAP
If chartType is "heatmap":
- For each visible cell where row/col labels are readable:
ex:cell-{IMG_ID}-{M} a chart:Cell ;
    chart:row "..." ;
    chart:col "..." ;
    chart:value "..."^^xsd:decimal ;
    chart:valueEstimated "true"^^xsd:boolean| "false"^^xsd:boolean .

8.8) TABLE
If chartType is "table":
- Output row/col/value triples:
ex:cell-{IMG_ID}-{M} a chart:Cell ;
    chart:row "..." ;
    chart:col "..." ;
    chart:value "..."^^xsd:decimal .

If numeric value is not readable, omit that cell.

========================
9) FINAL anti-empty rule
========================
- If the image clearly contains plotted data, output at least 3 datapoint nodes (marks/points/slices/bins/cells/stats).
- Only output zero datapoints if the image truly contains no readable plotted data, and in that case add:
  ex:chart-{IMG_ID} chart:extractionFailure "no_readable_datapoints" .

Now, for the chart image with id {IMG_ID}, output ONLY the RDF/Turtle following the rules above.
"""

# =========================
# 3) HELPERS
# =========================
def get_mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext in {".jpg", ".jpeg"}: return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext in {".tif", ".tiff"}: return "image/tiff"
    if ext == ".webp": return "image/webp"
    if ext == ".bmp": return "image/bmp"
    return "image/jpeg"

def encode_image_to_base64(image_path: str) -> Optional[str]:
    try:
        with Image.open(image_path) as _:
            pass  # Just to verify it's a valid image
        data = Path(image_path).read_bytes()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        print(f"[WARN] File not found: {image_path}")
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {image_path}: {e}")
        return None

def build_messages(image_b64: str, mime_type: str, prompt: str, extra_text: Optional[str] = None) -> list:
    """
    OpenAI-compatible messages for a vision request.
    image_b64 -> data URI inside an image_url content block.
    """
    content = [
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_b64}"
            }
        },
        {
            "type": "text",
            "text": prompt + ("\n\n" + extra_text if extra_text else "")
        }
    ]
    return [{"role": "user", "content": content}]

def _strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1 and t[:first_newline].startswith("```"):
            t = t[first_newline + 1 :]
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t.strip()

def call_avalai(messages: list, max_retries: int = 4) -> Optional[str]:
    """POST to AVALAI (OpenAI-compatible endpoint) with retry + back-off."""
    url     = f"{BASE_URL}/chat/completions"
    headers = {
        "Content-Type":  "application/json",
        "Authorization": f"Bearer {AVALAI_API_KEY}",
    }
    payload = {
        "model":    MODEL,
        "messages": messages,
        "temperature":      0.1,
        "top_p":            0.9,
        "max_completion_tokens": 8192,
    }
    backoff = 4.0  # Start with 4 seconds for initial 429 errors
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code >= 500 or resp.status_code == 429:
                print(f"[INFO] Retryable HTTP {resp.status_code}; attempt {attempt+1}/{max_retries}")
                time.sleep(backoff)
                backoff *= 2.0  # Exponential backoff: 4s -> 8s -> 16s -> 32s
                continue
            resp.raise_for_status()
            data = resp.json()
            raw = data["choices"][0]["message"]["content"]
            result = _strip_code_fences(raw)
            # Add delay after successful request to respect rate limits
            time.sleep(REQUEST_DELAY)
            return result
        except requests.exceptions.RequestException as e:
            print(f"[WARN] Request error: {e}; attempt {attempt+1}/{max_retries}")
            time.sleep(backoff)
            backoff *= 2.0
    return None

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def list_images(root: str) -> List[str]:
    """Return list of image paths under IMAGES_ROOT."""
    paths: List[str] = []
    for p in Path(root).rglob("*"):
        if p.suffix.lower() in VALID_IMAGE_EXTS:
            paths.append(str(p))
    return sorted(paths)

def load_label_for_image(img_stem: str) -> Optional[Dict]:
    """Load the JSON label with the same stem as image: LABELS_ROOT/<stem>.json"""
    json_path = Path(LABELS_ROOT) / f"{img_stem}.json"
    if not json_path.exists():
        print(f"[WARN] No JSON label found for image id={img_stem} at {json_path}")
        return None
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] Failed to load JSON for image id={img_stem}: {e}")
        return None

# =========================
# 4) MAIN
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    image_paths = list_images(IMAGES_ROOT)
    print(f"[INFO] Found {len(image_paths)} images under: {IMAGES_ROOT}")

    manifest = []

    for img_path in image_paths:
        img_path_obj = Path(img_path)
        img_id = img_path_obj.stem  # e.g., "72" from "72.png"

        ttl_name = f"{img_id}.ttl"
        ttl_path = os.path.join(OUTPUT_DIR, ttl_name)

        if SKIP_EXISTING and Path(ttl_path).exists():
            print(f"[SKIP] TTL already exists: {ttl_path}")
            continue

        print(f"[INFO] Processing img_id={img_id} -> {ttl_path}")

        image_b64 = encode_image_to_base64(img_path)
        if not image_b64:
            print(f"[SKIP] Could not encode {img_path}")
            continue

        row = load_label_for_image(img_id)

        prompt = PROMPT_TEMPLATE.replace("{IMG_ID}", str(img_id))

        extra_text = None
        if USE_VIS_TEXT_CONTEXT and isinstance(row, dict):
            bits = []
            if row.get("caption_L1"):
                bits.append("CAPTION_L1: " + str(row["caption_L1"]))
            if row.get("caption_L2L3"):
                bits.append("CAPTION_L2L3: " + str(row["caption_L2L3"]))
            if row.get("scenegraph"):
                bits.append("SCENEGRAPH (abridged): " + str(row["scenegraph"])[:1200])
            if row.get("datatable"):
                bits.append("DATATABLE (abridged): " + str(row["datatable"])[:1200])
            if bits:
                extra_text = "\n".join(bits)

        messages = build_messages(image_b64, get_mime_type(img_path), prompt, extra_text=extra_text)
        ttl_text = call_avalai(messages)
        if not ttl_text:
            print(f"[WARN] No TTL returned for img_id={img_id}")
            continue

        with open(ttl_path, "w", encoding="utf-8") as f:
            f.write(ttl_text if ttl_text.endswith("\n") else ttl_text + "\n")

        manifest.append({
            "img_id": img_id,
            "source_image": img_path,
            "ttl_file": ttl_path,
            "json_label": str(Path(LABELS_ROOT) / f"{img_id}.json"),
        })

    if WRITE_MANIFEST:
        ensure_dir(OUTPUT_DIR)
        with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
            json.dump({"items": manifest}, mf, indent=2, ensure_ascii=False)

    print(f"[DONE] Wrote {len(manifest)} TTL files to: {OUTPUT_DIR}")
    if WRITE_MANIFEST:
        print(f"[INFO] Manifest: {MANIFEST_PATH}")

if __name__ == "__main__":
    main()

[INFO] Found 1000 images under: ./sub_image_1000
[INFO] Processing img_id=100 -> ./vistext_Zeroshot_outputs/100.ttl
[INFO] Retryable HTTP 429; attempt 1/4
[INFO] Retryable HTTP 429; attempt 2/4
[INFO] Retryable HTTP 429; attempt 3/4
[INFO] Retryable HTTP 429; attempt 4/4


KeyboardInterrupt: 

In [11]:
!zip -r /kaggle/working/vistext_Zeroshot_outputs.zip /kaggle/working/vistext_Zeroshot_outputs

  adding: kaggle/working/vistext_Zeroshot_outputs/ (stored 0%)
  adding: kaggle/working/vistext_Zeroshot_outputs/6321.ttl (deflated 85%)
  adding: kaggle/working/vistext_Zeroshot_outputs/8276.ttl (deflated 84%)
  adding: kaggle/working/vistext_Zeroshot_outputs/1248.ttl (deflated 86%)
  adding: kaggle/working/vistext_Zeroshot_outputs/3886.ttl (deflated 85%)
  adding: kaggle/working/vistext_Zeroshot_outputs/manifest.json (deflated 88%)
  adding: kaggle/working/vistext_Zeroshot_outputs/6696.ttl (deflated 83%)
  adding: kaggle/working/vistext_Zeroshot_outputs/2392.ttl (deflated 87%)
  adding: kaggle/working/vistext_Zeroshot_outputs/1046.ttl (deflated 86%)
  adding: kaggle/working/vistext_Zeroshot_outputs/6390.ttl (deflated 84%)
  adding: kaggle/working/vistext_Zeroshot_outputs/4491.ttl (deflated 87%)
  adding: kaggle/working/vistext_Zeroshot_outputs/421.ttl (deflated 83%)
  adding: kaggle/working/vistext_Zeroshot_outputs/6581.ttl (deflated 77%)
  adding: kaggle/working/vistext_Zeroshot_out

# OneShot prompt

In [27]:
import base64
import json
import os
import time
from pathlib import Path
from typing import Optional, Dict, List
import requests
from PIL import Image

# =========================
# 1) CONFIGURATION (AVALAI)
# =========================
AVALAI_API_KEY = os.getenv("AVALAI_API_KEY")
if not AVALAI_API_KEY:
    AVALAI_API_KEY = input("Enter AVALAI_API_KEY (or set env var AVALAI_API_KEY): ").strip()
if not AVALAI_API_KEY:
    raise RuntimeError("Missing API key. Set env var AVALAI_API_KEY or provide it when prompted.")

MODEL = "gemini-2.5-flash-lite"                 # ← مدلی که می‌خوای استفاده کنی
BASE_URL = "https://api.avalai.ir/v1"          # ← بیس URL ای AVALAI

# ---- THREE PATHS ----
IMAGES_ROOT = "./sub-image"        # 1) folder with images (relative to notebook)
LABELS_ROOT = "./ground truth"     # 2) folder with JSON labels (72.json etc.)
OUTPUT_DIR  = "./vistext_Oneshot_outputs"  # 3) output TTL folder (relative)

# ---- RATE LIMITING ----
REQUEST_DELAY = 0  # seconds between successful API calls (to respect rate limits)

# Optional flags
SKIP_EXISTING = True             # skip if TTL already exists
USE_VIS_TEXT_CONTEXT = False     # if True, send caption/scenegraph/datatable from JSON

WRITE_MANIFEST = True
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

VALID_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# =========================
# 2) PROMPT (One-shot here)
# =========================
PROMPT_TEMPLATE = r"""
System
You are a chart-to-RDF extractor. Given ONE chart image, output ONLY valid RDF/Turtle (TTL) describing the chart and its datapoints that are visible or can be reasonably estimated from axis ticks/gridlines.

The dataset contains many chart types (bar/line/area/scatter/pie/histogram/boxplot/heatmap/table/other). Your goal is HIGH RECALL without hallucinating. If values are not explicitly printed but can be estimated from the axes, you MUST still output them and mark them as estimated.

========================
1) Output format (STRICT)
========================
- Output Turtle only. No prose, no explanations, no code fences, no comments.
- Ensure syntactically valid Turtle.
- Escape quotes inside literals.
- Collapse extra whitespace in extracted text.

========================
2) Prefixes (YOU MAY USE)
========================
Use ONLY these prefixes at the top (exactly these 4):

@prefix :      <http://example.org/chart-extraction/> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .
@prefix rdfs:  <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema:<https://schema.org/> .

Do not introduce any other prefixes.

================================
3) Deterministic IDs (REQUIRED)
================================
Use the numeric image id {IMG_ID} in identifiers:

- Chart node:      :Chart{IMG_ID}
- Datapoints:      :DataPoint{IMG_ID}_{M}     (M = 1,2,3,...)

Do not invent other naming patterns.

=====================================
4) Required chart skeleton (ALWAYS)
=====================================
Always output a chart node with:
- a chart type class (choose one):
  schema:BarChart | schema:LineChart | schema:ScatterPlot | schema:PieChart | schema:Table | schema:CreativeWork
- schema:title if visible (omit if not readable)
- :hasXAxisLabel and :hasYAxisLabel (use "" if not readable)
- OPTIONAL: :xMin :xMax :yMin :yMax as xsd:decimal when clearly readable (otherwise omit)

Example pattern:
:Chart{IMG_ID} a schema:BarChart ;
    schema:title "..." ;
    :hasXAxisLabel "..." ;
    :hasYAxisLabel "..." .

=====================================
5) Datapoint model (UNIFIED FOR ALL CHART TYPES)
=====================================
You MUST output datapoints using the same structure for every chart type to avoid empty outputs:

Each datapoint MUST:
- belong to the chart using :belongsToChart
- have a readable label (rdfs:label) whenever possible
- have numeric value(s) when possible

Use these predicates:
- rdfs:label "..."                       (category/series/point label)
- :xValue "..."                          (string for x category/time/bin/column)
- :yValue "0.0"^^xsd:decimal             (main numeric value)
- :valueEstimated "true"^^xsd:boolean | "false"^^xsd:boolean
- :belongsToChart :Chart{IMG_ID}

Rules:
- If the chart is BAR: set rdfs:label to the category, yValue to the bar height/value. xValue can equal the category too.
- If the chart is LINE/AREA: set xValue to the tick label (year/month/category) and yValue to the value at that x.
- If the chart is SCATTER: each point is one datapoint with xValue and yValue (xValue can be the numeric x as a string).
- If the chart is PIE: rdfs:label = slice label, yValue = percent (0–100) and set :yUnit "percent" on the chart if visible.
- If the chart is HISTOGRAM: xValue = bin range (e.g., "10–20"), yValue = count/frequency.
- If the chart is HEATMAP/TABLE: xValue = column label, rdfs:label or :rowValue = row label, yValue = cell value (if numeric).
- If you cannot read xValue/labels, use placeholders like "UNREADABLE_{M}" ONLY if the datapoint is clearly present.

Estimation:
- If the number is explicitly printed (data label), set :valueEstimated "false".
- If you infer it from axis ticks/gridlines, set :valueEstimated "true".
- Do NOT guess wildly; use estimation only when the axis scale is visible.

========================
6) Anti-empty requirement
========================
- If the image clearly contains plotted/tabular data, you MUST output at least 3 datapoints.
- Only output zero datapoints if the image truly contains no readable datapoints, and then add:
  :Chart{IMG_ID} :extractionFailure "no_readable_datapoints" .

========================================
7) One-shot Example (Bar chart) 
========================================
Follow the style and structure of this example:

@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

ex:chart-1088 a chart:Chart ;
    chart:chartType "bar" ;
    chart:title "Given the current state of the business what are the CEO 's top three priorities for you to help business preserve through the current disruption ?" ;
    chart:xAxis ex:x-1088 ;
    chart:yAxis ex:y-1088 .

ex:x-1088 a chart:Axis ;
    chart:label "Response" .

ex:y-1088 a chart:Axis ;
    chart:label "" .

ex:series-1088-1 a chart:Series .

ex:mark-1088-1-1 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Share of respondents Lead digital buisness/digital..." ;
    chart:value "0.37"^^xsd:decimal .

ex:mark-1088-1-2 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Improve remote work experiences" ;
    chart:value "0.37"^^xsd:decimal .

ex:mark-1088-1-3 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Upgrade IT and data security to boost corporate re..." ;
    chart:value "0.29"^^xsd:decimal .

ex:mark-1088-1-4 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Reduce or rationalize IT spending" ;
    chart:value "0.28"^^xsd:decimal .

ex:mark-1088-1-5 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Help reach specific goal for corporate revenue gro..." ;
    chart:value "0.25"^^xsd:decimal .

ex:mark-1088-1-6 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Identify new data-driven business opportunities" ;
    chart:value "0.24"^^xsd:decimal .

ex:mark-1088-1-7 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Enable new plan for customer acquisition & retenti..." ;
    chart:value "0.23"^^xsd:decimal .

ex:mark-1088-1-8 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Collaborate with business departments om major cus..." ;
    chart:value "0.22"^^xsd:decimal .

ex:mark-1088-1-9 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Lead a product innovation effort" ;
    chart:value "0.14"^^xsd:decimal .

ex:mark-1088-1-10 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Increases collaboration with LOB to help streamlin..." ;
    chart:value "0.11"^^xsd:decimal .

ex:mark-1088-1-11 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Lead merger integration or due dilligence on a pot..." ;
    chart:value "0.08"^^xsd:decimal .


========================================
8) Final instruction
========================================
Now, for the current chart image with id {IMG_ID}, output ONLY RDF/Turtle (TTL).
- Use deterministic IDs :Chart{IMG_ID} and :DataPoint{IMG_ID}_{M}.
- Use the unified datapoint model from section 5 (include :valueEstimated).
- Keep output parseable Turtle.
Do not output any explanations.
"""

# =========================
# 3) HELPERS
# =========================
def get_mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext in {".jpg", ".jpeg"}: return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext in {".tif", ".tiff"}: return "image/tiff"
    if ext == ".webp": return "image/webp"
    if ext == ".bmp": return "image/bmp"
    return "image/jpeg"

def encode_image_to_base64(image_path: str) -> Optional[str]:
    try:
        with Image.open(image_path) as _:
            pass  # Just to verify it's a valid image
        data = Path(image_path).read_bytes()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        print(f"[WARN] File not found: {image_path}")
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {image_path}: {e}")
        return None

def build_messages(image_b64: str, mime_type: str, prompt: str, extra_text: Optional[str] = None) -> list:
    """
    OpenAI-compatible messages for a vision request.
    image_b64 -> data URI inside an image_url content block.
    """
    content = [
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_b64}"
            }
        },
        {
            "type": "text",
            "text": prompt + ("\n\n" + extra_text if extra_text else "")
        }
    ]
    return [{"role": "user", "content": content}]

def _strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1 and t[:first_newline].startswith("```"):
            t = t[first_newline + 1 :]
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t.strip()

def call_avalai(messages: list, max_retries: int = 4) -> Optional[str]:
    """POST to AVALAI (OpenAI-compatible endpoint) with retry + back-off."""
    url     = f"{BASE_URL}/chat/completions"
    headers = {
        "Content-Type":  "application/json",
        "Authorization": f"Bearer {AVALAI_API_KEY}",
    }
    payload = {
        "model":    MODEL,
        "messages": messages,
        "temperature":      0.1,
        "top_p":            0.9,
        "max_completion_tokens": 8192,
    }
    backoff = 4.0  # Start with 4 seconds for initial 429 errors
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code >= 500 or resp.status_code == 429:
                print(f"[INFO] Retryable HTTP {resp.status_code}; attempt {attempt+1}/{max_retries}")
                time.sleep(backoff)
                backoff *= 2.0  # Exponential backoff: 4s -> 8s -> 16s -> 32s
                continue
            resp.raise_for_status()
            data = resp.json()
            raw = data["choices"][0]["message"]["content"]
            result = _strip_code_fences(raw)
            # Add delay after successful request to respect rate limits
            time.sleep(REQUEST_DELAY)
            return result
        except requests.exceptions.RequestException as e:
            print(f"[WARN] Request error: {e}; attempt {attempt+1}/{max_retries}")
            time.sleep(backoff)
            backoff *= 2.0
    return None

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def list_images(root: str) -> List[str]:
    """Return list of image paths under IMAGES_ROOT."""
    paths: List[str] = []
    for p in Path(root).rglob("*"):
        if p.suffix.lower() in VALID_IMAGE_EXTS:
            paths.append(str(p))
    return sorted(paths)

def load_label_for_image(img_stem: str) -> Optional[Dict]:
    """Load the JSON label with the same stem as image: LABELS_ROOT/<stem>.json"""
    json_path = Path(LABELS_ROOT) / f"{img_stem}.json"
    if not json_path.exists():
        print(f"[WARN] No JSON label found for image id={img_stem} at {json_path}")
        return None
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] Failed to load JSON for image id={img_stem}: {e}")
        return None

# =========================
# 4) MAIN
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    image_paths = list_images(IMAGES_ROOT)
    print(f"[INFO] Found {len(image_paths)} images under: {IMAGES_ROOT}")

    manifest = []

    for img_path in image_paths:
        img_path_obj = Path(img_path)
        img_id = img_path_obj.stem  # e.g., "72" from "72.png"

        ttl_name = f"{img_id}.ttl"
        ttl_path = os.path.join(OUTPUT_DIR, ttl_name)

        if SKIP_EXISTING and Path(ttl_path).exists():
            print(f"[SKIP] TTL already exists: {ttl_path}")
            continue

        print(f"[INFO] Processing img_id={img_id} -> {ttl_path}")

        image_b64 = encode_image_to_base64(img_path)
        if not image_b64:
            print(f"[SKIP] Could not encode {img_path}")
            continue

        row = load_label_for_image(img_id)

        prompt = PROMPT_TEMPLATE.replace("{IMG_ID}", str(img_id))

        extra_text = None
        if USE_VIS_TEXT_CONTEXT and isinstance(row, dict):
            bits = []
            if row.get("caption_L1"):
                bits.append("CAPTION_L1: " + str(row["caption_L1"]))
            if row.get("caption_L2L3"):
                bits.append("CAPTION_L2L3: " + str(row["caption_L2L3"]))
            if row.get("scenegraph"):
                bits.append("SCENEGRAPH (abridged): " + str(row["scenegraph"])[:1200])
            if row.get("datatable"):
                bits.append("DATATABLE (abridged): " + str(row["datatable"])[:1200])
            if bits:
                extra_text = "\n".join(bits)

        messages = build_messages(image_b64, get_mime_type(img_path), prompt, extra_text=extra_text)
        ttl_text = call_avalai(messages)
        if not ttl_text:
            print(f"[WARN] No TTL returned for img_id={img_id}")
            continue

        with open(ttl_path, "w", encoding="utf-8") as f:
            f.write(ttl_text if ttl_text.endswith("\n") else ttl_text + "\n")

        manifest.append({
            "img_id": img_id,
            "source_image": img_path,
            "ttl_file": ttl_path,
            "json_label": str(Path(LABELS_ROOT) / f"{img_id}.json"),
        })

    if WRITE_MANIFEST:
        ensure_dir(OUTPUT_DIR)
        with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
            json.dump({"items": manifest}, mf, indent=2, ensure_ascii=False)

    print(f"[DONE] Wrote {len(manifest)} TTL files to: {OUTPUT_DIR}")
    if WRITE_MANIFEST:
        print(f"[INFO] Manifest: {MANIFEST_PATH}")

if __name__ == "__main__":
    main()

[INFO] Found 100 images under: ./sub-image
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1046.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1088.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1093.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1150.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1168.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1170.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1248.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1404.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1515.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1752.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1793.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1808.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/1894.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/2094.ttl
[SKIP] TTL already exists: ./vistext_Oneshot_outputs/2141.ttl
[SKIP] TTL already exists: 

In [22]:
!zip -r /kaggle/working/vistext_Oneshot_outputs.zip /kaggle/working/vistext_Oneshot_outputs

  adding: kaggle/working/vistext_Oneshot_outputs/ (stored 0%)
  adding: kaggle/working/vistext_Oneshot_outputs/6321.ttl (deflated 82%)
  adding: kaggle/working/vistext_Oneshot_outputs/8276.ttl (deflated 82%)
  adding: kaggle/working/vistext_Oneshot_outputs/1248.ttl (deflated 85%)
  adding: kaggle/working/vistext_Oneshot_outputs/3886.ttl (deflated 82%)
  adding: kaggle/working/vistext_Oneshot_outputs/manifest.json (deflated 90%)
  adding: kaggle/working/vistext_Oneshot_outputs/6696.ttl (deflated 82%)
  adding: kaggle/working/vistext_Oneshot_outputs/2392.ttl (deflated 86%)
  adding: kaggle/working/vistext_Oneshot_outputs/1046.ttl (deflated 85%)
  adding: kaggle/working/vistext_Oneshot_outputs/6390.ttl (deflated 82%)
  adding: kaggle/working/vistext_Oneshot_outputs/4491.ttl (deflated 86%)
  adding: kaggle/working/vistext_Oneshot_outputs/421.ttl (deflated 83%)
  adding: kaggle/working/vistext_Oneshot_outputs/6581.ttl (deflated 75%)
  adding: kaggle/working/vistext_Oneshot_outputs/8496.ttl 

# FewShot Prompting

In [28]:
import base64
import json
import os
import time
from pathlib import Path
from typing import Optional, Dict, List
import requests
from PIL import Image

# =========================
# 1) CONFIGURATION (AVALAI)
# =========================
AVALAI_API_KEY = os.getenv("AVALAI_API_KEY")
if not AVALAI_API_KEY:
    AVALAI_API_KEY = input("Enter AVALAI_API_KEY (or set env var AVALAI_API_KEY): ").strip()
if not AVALAI_API_KEY:
    raise RuntimeError("Missing API key. Set env var AVALAI_API_KEY or provide it when prompted.")

MODEL = "gemini-2.5-flash-lite"                 # ← مدلی که می‌خوای استفاده کنی
BASE_URL = "https://api.avalai.ir/v1"          # ← بیس URL ای AVALAI

# ---- THREE PATHS ----
IMAGES_ROOT = "./sub-image"        # 1) folder with images (relative to notebook)
LABELS_ROOT = "./ground truth"     # 2) folder with JSON labels (72.json etc.)
OUTPUT_DIR  = "./vistext_Fewshot_outputs"  # 3) output TTL folder (relative)

# ---- RATE LIMITING ----
REQUEST_DELAY = 0  # seconds between successful API calls (to respect rate limits)

# Optional flags
SKIP_EXISTING = True             # skip if TTL already exists
USE_VIS_TEXT_CONTEXT = False     # if True, send caption/scenegraph/datatable from JSON

WRITE_MANIFEST = True
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

VALID_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# =========================
# 2) PROMPT (Few-shot here)
# =========================
PROMPT_TEMPLATE = r"""

System
You are a chart-to-RDF extractor. Given ONE chart image, output ONLY valid RDF/Turtle (TTL) capturing:
- chart metadata (type, title, axes)
- series (if any)
- datapoints with HIGH RECALL

The dataset contains many chart types (bar/stacked_bar/line/area/scatter/pie/histogram/boxplot/heatmap/table/other).
Primary objective: minimize "zero datapoints" and maximize correct category coverage.
If values are not explicitly printed but can be estimated from axis ticks/gridlines, you MUST output them and mark them as estimated.

========================
1) Output format (STRICT)
========================
- Output Turtle ONLY. No explanations, no prose, no code fences, no comments.
- Must be syntactically valid Turtle.
- Escape quotes in string literals.
- Preserve readable text as-is (do NOT shorten labels using "...").
- If you are unsure about a value, estimate it only if the axis scale is visible and mark it as estimated.

========================
2) Prefixes (ONLY THESE)
========================
Use exactly these prefixes (and no others):

@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

================================
3) Deterministic identifiers (ONLY)
================================
Use the numeric image id {IMG_ID} inside IRIs exactly like this:
- Chart:      ex:chart-{IMG_ID}
- X axis:     ex:x-{IMG_ID}
- Y axis:     ex:y-{IMG_ID}
- Series:     ex:series-{IMG_ID}-{N}          (N=1,2,3,...)
- DataPoints: ex:dp-{IMG_ID}-{N}-{M}          (series N, index M)

Do not invent other naming patterns.

=================================
4) REQUIRED base graph (ALWAYS)
=================================
You MUST always output the skeleton below (fill literals when visible; otherwise keep ""):

ex:chart-{IMG_ID} a chart:Chart ;
    chart:chartType "bar"|"stacked_bar"|"line"|"area"|"scatter"|"pie"|"histogram"|"boxplot"|"heatmap"|"table"|"other" ;
    chart:xAxis ex:x-{IMG_ID} ;
    chart:yAxis ex:y-{IMG_ID} .

ex:x-{IMG_ID} a chart:Axis ;
    chart:label "" ;
    chart:scaleMin "" ;
    chart:scaleMax "" .

ex:y-{IMG_ID} a chart:Axis ;
    chart:label "" ;
    chart:scaleMin "" ;
    chart:scaleMax "" .

# Always create at least one series:
ex:series-{IMG_ID}-1 a chart:Series .

======================================
5) Chart metadata extraction (if visible)
======================================
- Title: if readable, add:
  ex:chart-{IMG_ID} chart:title "..." .
- Axis labels: if readable, set chart:label on ex:x-{IMG_ID} and ex:y-{IMG_ID}.
- Axis min/max ticks: if readable, set chart:scaleMin and chart:scaleMax (plain strings, no commas).
- If multiple series exist (legend/colors/line styles), create multiple series nodes and (if readable) add:
  ex:series-{IMG_ID}-{N} chart:seriesName "..." .

===========================================
6) Datapoints: unified schema (ALL chart types)
===========================================
Always represent datapoints using chart:DataPoint nodes.
This avoids "empty outputs" across chart types.

Each datapoint MUST:
- belong to exactly one series
- belong to the chart via the series relationship
- include a label/category or an x-position (or both)
- include a numeric value when possible (printed OR estimated)

Datapoint core predicates:
- chart:series ex:series-{IMG_ID}-{N}
- chart:category "..."                # for categorical x / slice label / bin range / table cell label
- chart:value "..."^^xsd:decimal      # primary numeric value
- chart:xVal "..."                    # for numeric/time x when useful (string)
- chart:yVal "..."^^xsd:decimal       # for line/scatter y when useful
- chart:valueEstimated "true"^^xsd:boolean | "false"^^xsd:boolean

Rules by chart type:
- bar/stacked_bar: use chart:category + chart:value
- line/area: use chart:xVal + chart:yVal (also you MAY set chart:value equal to yVal)
- scatter: use chart:xVal + chart:yVal
- pie: use chart:category + chart:value (value is percent if chart shows percent; otherwise fraction/absolute if clearly shown)
- histogram: use chart:category for bin range (e.g., "10–20") + chart:value count
- heatmap/table: use chart:category for "row|col" or similar + chart:value if numeric

Estimation:
- If number is explicitly printed -> chart:valueEstimated "false"
- If inferred from axis ticks/gridlines -> chart:valueEstimated "true"
- Do NOT omit datapoints just because values are not printed; estimate when scale is readable.

========================
7) Anti-empty requirement
========================
- If the image clearly contains plotted/tabular data, you MUST output at least 3 datapoints.
- Only output zero datapoints if the image truly contains no readable datapoints at all, and then add:
  ex:chart-{IMG_ID} chart:extractionFailure "no_readable_datapoints" .

========================
8) Few-shot Examples (STYLE GUIDE)
========================
These examples show the exact TTL style you must follow (prefixes, IRIs, predicates, grouping).
Do NOT copy their values blindly—extract from the current image.

-------------------------
Example 1 (Survey Bar Chart, derived from 1088-style)
-------------------------
@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

ex:chart-1088 a chart:Chart ;
    chart:chartType "bar" ;
    chart:title "Given the current state of the business what are the CEO 's top three priorities for you to help business preserve through the current disruption ?" ;
    chart:xAxis ex:x-1088 ;
    chart:yAxis ex:y-1088 .

ex:x-1088 a chart:Axis ;
    chart:label "Response" .

ex:y-1088 a chart:Axis ;
    chart:label "" .

ex:series-1088-1 a chart:Series .

ex:mark-1088-1-1 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Share of respondents Lead digital buisness/digital..." ;
    chart:value "0.37"^^xsd:decimal .

ex:mark-1088-1-2 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Improve remote work experiences" ;
    chart:value "0.37"^^xsd:decimal .

ex:mark-1088-1-3 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Upgrade IT and data security to boost corporate re..." ;
    chart:value "0.29"^^xsd:decimal .

ex:mark-1088-1-4 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Reduce or rationalize IT spending" ;
    chart:value "0.28"^^xsd:decimal .

ex:mark-1088-1-5 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Help reach specific goal for corporate revenue gro..." ;
    chart:value "0.25"^^xsd:decimal .

ex:mark-1088-1-6 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Identify new data-driven business opportunities" ;
    chart:value "0.24"^^xsd:decimal .

ex:mark-1088-1-7 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Enable new plan for customer acquisition & retenti..." ;
    chart:value "0.23"^^xsd:decimal .

ex:mark-1088-1-8 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Collaborate with business departments om major cus..." ;
    chart:value "0.22"^^xsd:decimal .

ex:mark-1088-1-9 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Lead a product innovation effort" ;
    chart:value "0.14"^^xsd:decimal .

ex:mark-1088-1-10 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Increases collaboration with LOB to help streamlin..." ;
    chart:value "0.11"^^xsd:decimal .

ex:mark-1088-1-11 a chart:DataPoint ;
    chart:series ex:series-1088-1 ;
    chart:category "Lead merger integration or due dilligence on a pot..." ;
    chart:value "0.08"^^xsd:decimal .

-------------------------
Example 2 (Bar Chart, id=76)
-------------------------
@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

ex:chart-76 a chart:Chart ;
    chart:chartType "bar" ;
    chart:title "Non-English main home languages ranked by number of speakers in Welsh schools in 2020" ;
    chart:xAxis ex:x-76 ;
    chart:yAxis ex:y-76 .

ex:x-76 a chart:Axis ;
    chart:label "geographic region" .

ex:y-76 a chart:Axis ;
    chart:label "" .

ex:series-76-1 a chart:Series .

ex:mark-76-1-1 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "region Number of speakers Polish" ;
    chart:value "4961"^^xsd:decimal .

ex:mark-76-1-2 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Arabic" ;
    chart:value "3484"^^xsd:decimal .

ex:mark-76-1-3 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Bengali" ;
    chart:value "3019"^^xsd:decimal .

ex:mark-76-1-4 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Urdu" ;
    chart:value "1508"^^xsd:decimal .

ex:mark-76-1-5 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Chinese" ;
    chart:value "1466"^^xsd:decimal .

ex:mark-76-1-6 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Panjabi" ;
    chart:value "1242"^^xsd:decimal .

ex:mark-76-1-7 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Romanian" ;
    chart:value "1091"^^xsd:decimal .

ex:mark-76-1-8 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Somali" ;
    chart:value "852"^^xsd:decimal .

ex:mark-76-1-9 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Portuguese" ;
    chart:value "846"^^xsd:decimal .

ex:mark-76-1-10 a chart:DataPoint ;
    chart:series ex:series-76-1 ;
    chart:category "Tagalog/Filipino" ;
    chart:value "830"^^xsd:decimal .


-------------------------
Example 3 (Bar Chart, id=344)
-------------------------
@prefix ex:    <http://example.org/> .
@prefix chart: <http://example.org/chart#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .

ex:chart-344 a chart:Chart ;
    chart:chartType "bar" ;
    chart:title "Revenue of the New York Islanders from 2005/06 to 2018/19 (in million U.S. dollars)" ;
    chart:xAxis ex:x-344 ;
    chart:yAxis ex:y-344 .

ex:x-344 a chart:Axis ;
    chart:label "Revenue in million U" .

ex:y-344 a chart:Axis ;
    chart:label "Year" .

ex:series-344-1 a chart:Series .

ex:mark-344-1-1 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2018/19" ;
    chart:value "107"^^xsd:decimal .

ex:mark-344-1-2 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2017/18" ;
    chart:value "110"^^xsd:decimal .

ex:mark-344-1-3 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2016/17" ;
    chart:value "114"^^xsd:decimal .

ex:mark-344-1-4 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2015/16" ;
    chart:value "93"^^xsd:decimal .

ex:mark-344-1-5 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2014/15" ;
    chart:value "83"^^xsd:decimal .

ex:mark-344-1-6 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2013/14" ;
    chart:value "61"^^xsd:decimal .

ex:mark-344-1-7 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2012/13*" ;
    chart:value "66"^^xsd:decimal .

ex:mark-344-1-8 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2011/12" ;
    chart:value "63"^^xsd:decimal .

ex:mark-344-1-9 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2010/11" ;
    chart:value "63"^^xsd:decimal .

ex:mark-344-1-10 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2009/10" ;
    chart:value "62"^^xsd:decimal .

ex:mark-344-1-11 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2008/09" ;
    chart:value "64"^^xsd:decimal .

ex:mark-344-1-12 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2007/08" ;
    chart:value "60"^^xsd:decimal .

ex:mark-344-1-13 a chart:DataPoint ;
    chart:series ex:series-344-1 ;
    chart:category "2006/07" ;
    chart:value "56"^^xsd:decimal .

    
========================================
9) FINAL instruction for the current image
========================================
Now, for the chart image with id {IMG_ID}, output ONLY RDF/Turtle (TTL) using:
- the REQUIRED skeleton (section 4)
- the unified datapoint model (section 6)
- anti-empty rule (section 7)

Do not shorten labels using "...".
If values are not printed but axes are readable, estimate values and set chart:valueEstimated "true".
Output only Turtle.

"""

# =========================
# 3) HELPERS
# =========================
def get_mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext in {".jpg", ".jpeg"}: return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext in {".tif", ".tiff"}: return "image/tiff"
    if ext == ".webp": return "image/webp"
    if ext == ".bmp": return "image/bmp"
    return "image/jpeg"

def encode_image_to_base64(image_path: str) -> Optional[str]:
    try:
        with Image.open(image_path) as _:
            pass  # Just to verify it's a valid image
        data = Path(image_path).read_bytes()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        print(f"[WARN] File not found: {image_path}")
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {image_path}: {e}")
        return None

def build_messages(image_b64: str, mime_type: str, prompt: str, extra_text: Optional[str] = None) -> list:
    """
    OpenAI-compatible messages for a vision request.
    image_b64 -> data URI inside an image_url content block.
    """
    content = [
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_b64}"
            }
        },
        {
            "type": "text",
            "text": prompt + ("\n\n" + extra_text if extra_text else "")
        }
    ]
    return [{"role": "user", "content": content}]

def _strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1 and t[:first_newline].startswith("```"):
            t = t[first_newline + 1 :]
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t.strip()

def call_avalai(messages: list, max_retries: int = 4) -> Optional[str]:
    """POST to AVALAI (OpenAI-compatible endpoint) with retry + back-off."""
    url     = f"{BASE_URL}/chat/completions"
    headers = {
        "Content-Type":  "application/json",
        "Authorization": f"Bearer {AVALAI_API_KEY}",
    }
    payload = {
        "model":    MODEL,
        "messages": messages,
        "temperature":      0.1,
        "top_p":            0.9,
        "max_completion_tokens": 8192,
    }
    backoff = 4.0  # Start with 4 seconds for initial 429 errors
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code >= 500 or resp.status_code == 429:
                print(f"[INFO] Retryable HTTP {resp.status_code}; attempt {attempt+1}/{max_retries}")
                time.sleep(backoff)
                backoff *= 2.0  # Exponential backoff: 4s -> 8s -> 16s -> 32s
                continue
            resp.raise_for_status()
            data = resp.json()
            raw = data["choices"][0]["message"]["content"]
            result = _strip_code_fences(raw)
            # Add delay after successful request to respect rate limits
            time.sleep(REQUEST_DELAY)
            return result
        except requests.exceptions.RequestException as e:
            print(f"[WARN] Request error: {e}; attempt {attempt+1}/{max_retries}")
            time.sleep(backoff)
            backoff *= 2.0
    return None

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def list_images(root: str) -> List[str]:
    """Return list of image paths under IMAGES_ROOT."""
    paths: List[str] = []
    for p in Path(root).rglob("*"):
        if p.suffix.lower() in VALID_IMAGE_EXTS:
            paths.append(str(p))
    return sorted(paths)

def load_label_for_image(img_stem: str) -> Optional[Dict]:
    """Load the JSON label with the same stem as image: LABELS_ROOT/<stem>.json"""
    json_path = Path(LABELS_ROOT) / f"{img_stem}.json"
    if not json_path.exists():
        print(f"[WARN] No JSON label found for image id={img_stem} at {json_path}")
        return None
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] Failed to load JSON for image id={img_stem}: {e}")
        return None

# =========================
# 4) MAIN
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    image_paths = list_images(IMAGES_ROOT)
    print(f"[INFO] Found {len(image_paths)} images under: {IMAGES_ROOT}")

    manifest = []

    for img_path in image_paths:
        img_path_obj = Path(img_path)
        img_id = img_path_obj.stem  # e.g., "72" from "72.png"

        ttl_name = f"{img_id}.ttl"
        ttl_path = os.path.join(OUTPUT_DIR, ttl_name)

        if SKIP_EXISTING and Path(ttl_path).exists():
            print(f"[SKIP] TTL already exists: {ttl_path}")
            continue

        print(f"[INFO] Processing img_id={img_id} -> {ttl_path}")

        image_b64 = encode_image_to_base64(img_path)
        if not image_b64:
            print(f"[SKIP] Could not encode {img_path}")
            continue

        row = load_label_for_image(img_id)

        prompt = PROMPT_TEMPLATE.replace("{IMG_ID}", str(img_id))

        extra_text = None
        if USE_VIS_TEXT_CONTEXT and isinstance(row, dict):
            bits = []
            if row.get("caption_L1"):
                bits.append("CAPTION_L1: " + str(row["caption_L1"]))
            if row.get("caption_L2L3"):
                bits.append("CAPTION_L2L3: " + str(row["caption_L2L3"]))
            if row.get("scenegraph"):
                bits.append("SCENEGRAPH (abridged): " + str(row["scenegraph"])[:1200])
            if row.get("datatable"):
                bits.append("DATATABLE (abridged): " + str(row["datatable"])[:1200])
            if bits:
                extra_text = "\n".join(bits)

        messages = build_messages(image_b64, get_mime_type(img_path), prompt, extra_text=extra_text)
        ttl_text = call_avalai(messages)
        if not ttl_text:
            print(f"[WARN] No TTL returned for img_id={img_id}")
            continue

        with open(ttl_path, "w", encoding="utf-8") as f:
            f.write(ttl_text if ttl_text.endswith("\n") else ttl_text + "\n")

        manifest.append({
            "img_id": img_id,
            "source_image": img_path,
            "ttl_file": ttl_path,
            "json_label": str(Path(LABELS_ROOT) / f"{img_id}.json"),
        })

    if WRITE_MANIFEST:
        ensure_dir(OUTPUT_DIR)
        with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
            json.dump({"items": manifest}, mf, indent=2, ensure_ascii=False)

    print(f"[DONE] Wrote {len(manifest)} TTL files to: {OUTPUT_DIR}")
    if WRITE_MANIFEST:
        print(f"[INFO] Manifest: {MANIFEST_PATH}")

if __name__ == "__main__":
    main()

[INFO] Found 100 images under: ./sub-image
[INFO] Processing img_id=1046 -> ./vistext_Fewshot_outputs/1046.ttl
[INFO] Processing img_id=1088 -> ./vistext_Fewshot_outputs/1088.ttl
[INFO] Processing img_id=1093 -> ./vistext_Fewshot_outputs/1093.ttl
[INFO] Processing img_id=1150 -> ./vistext_Fewshot_outputs/1150.ttl
[INFO] Processing img_id=1168 -> ./vistext_Fewshot_outputs/1168.ttl
[INFO] Processing img_id=1170 -> ./vistext_Fewshot_outputs/1170.ttl
[INFO] Processing img_id=1248 -> ./vistext_Fewshot_outputs/1248.ttl
[INFO] Processing img_id=1404 -> ./vistext_Fewshot_outputs/1404.ttl
[INFO] Processing img_id=1515 -> ./vistext_Fewshot_outputs/1515.ttl
[INFO] Processing img_id=1752 -> ./vistext_Fewshot_outputs/1752.ttl
[INFO] Processing img_id=1793 -> ./vistext_Fewshot_outputs/1793.ttl
[INFO] Processing img_id=1808 -> ./vistext_Fewshot_outputs/1808.ttl
[INFO] Processing img_id=1894 -> ./vistext_Fewshot_outputs/1894.ttl
[INFO] Processing img_id=2094 -> ./vistext_Fewshot_outputs/2094.ttl
[INFO

In [31]:
!zip -r /kaggle/working/vistext_Fewshot_outputs.zip /kaggle/working/vistext_Fewshot_outputs

  adding: kaggle/working/vistext_Fewshot_outputs/ (stored 0%)
  adding: kaggle/working/vistext_Fewshot_outputs/6321.ttl (deflated 84%)
  adding: kaggle/working/vistext_Fewshot_outputs/8276.ttl (deflated 85%)
  adding: kaggle/working/vistext_Fewshot_outputs/1248.ttl (deflated 86%)
  adding: kaggle/working/vistext_Fewshot_outputs/3886.ttl (deflated 85%)
  adding: kaggle/working/vistext_Fewshot_outputs/manifest.json (deflated 90%)
  adding: kaggle/working/vistext_Fewshot_outputs/6696.ttl (deflated 85%)
  adding: kaggle/working/vistext_Fewshot_outputs/2392.ttl (deflated 87%)
  adding: kaggle/working/vistext_Fewshot_outputs/1046.ttl (deflated 86%)
  adding: kaggle/working/vistext_Fewshot_outputs/6390.ttl (deflated 84%)
  adding: kaggle/working/vistext_Fewshot_outputs/4491.ttl (deflated 87%)
  adding: kaggle/working/vistext_Fewshot_outputs/421.ttl (deflated 83%)
  adding: kaggle/working/vistext_Fewshot_outputs/6581.ttl (deflated 78%)
  adding: kaggle/working/vistext_Fewshot_outputs/8496.ttl 